<a href="https://colab.research.google.com/github/mscids2024pranita-hue/goa-rainfall-arima/blob/main/01_data_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Loading
**Project:** Goa Rainfall Forecasting using SARIMA
**Data:** IMD Gridded Daily Rainfall (0.25°), 1991–2025
**Step:** Install libraries, load one NetCDF file, understand its structure

In [1]:
# Install libraries not available by default in Colab
!pip install netCDF4 xarray -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.1 MB/s eta 0:00:00


In [2]:
# Import all libraries we need for data loading
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob

print("All libraries imported successfully!")
print(f"xarray version  : {xr.__version__}")
print(f"numpy version   : {np.__version__}")
print(f"pandas version  : {pd.__version__}")

All libraries imported successfully!
xarray version  : 2025.12.0
numpy version   : 2.0.2
pandas version  : 2.2.2


In [3]:
from google.colab import files

uploaded = files.upload()
# A file picker will appear — select your zip file

Saving imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip to imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip


In [4]:
import zipfile
import os

zip_path = "imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip"

# Unzip into a folder called 'imd_data'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("imd_data")

print("Unzipped successfully!")
print("\nContents of imd_data folder:")
for f in sorted(os.listdir("imd_data/imd_rainfall-20260513T063252Z-3-001/imd_rainfall/")):
    print(f"  {f}")

Unzipped successfully!

Contents of imd_data folder:
  RF25_ind1991_rfp25.nc
  RF25_ind1992_rfp25.nc
  RF25_ind1993_rfp25.nc
  RF25_ind1994_rfp25.nc
  RF25_ind1995_rfp25.nc
  RF25_ind1996_rfp25.nc
  RF25_ind1997_rfp25.nc
  RF25_ind1998_rfp25.nc
  RF25_ind1999_rfp25.nc
  RF25_ind2000_rfp25.nc
  RF25_ind2001_rfp25.nc
  RF25_ind2002_rfp25.nc
  RF25_ind2003_rfp25.nc
  RF25_ind2004_rfp25.nc
  RF25_ind2005_rfp25 (1).nc
  RF25_ind2006_rfp25.nc
  RF25_ind2007_rfp25.nc
  RF25_ind2008_rfp25.nc
  RF25_ind2009_rfp25.nc
  RF25_ind2010_rfp25.nc
  RF25_ind2011_rfp25.nc
  RF25_ind2012_rfp25.nc
  RF25_ind2013_rfp25.nc
  RF25_ind2014_rfp25.nc
  RF25_ind2015_rfp25.nc
  RF25_ind2016_rfp25.nc
  RF25_ind2017_rfp25.nc
  RF25_ind2018_rfp25.nc
  RF25_ind2019_rfp25.nc
  RF25_ind2020_rfp25.nc
  RF25_ind2021_rfp25.nc
  RF25_ind2022_rfp25.nc
  RF25_ind2023_rfp25.nc
  RF25_ind2024_rfp25.nc
  RF25_ind2025_rfp25.nc
  RF25_ind_rfp25 (1).nc
  RF25_ind_rfp25 (2).nc
  RF25_ind_rfp25 (3).nc
  RF25_ind_rfp25.nc


In [5]:
# Define the path to the data folder
DATA_DIR = "imd_data/imd_rainfall-20260513T063252Z-3-001/imd_rainfall/"

# Open just one file — 1991
ds = xr.open_dataset(DATA_DIR + "RF25_ind1991_rfp25.nc")

# Print the full structure
print(ds)

<xarray.Dataset> Size: 25MB
Dimensions:    (TIME: 365, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 1991-01-01 1991-01-02 ... 1991-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 25MB ...
Attributes:
    history:      FERRET V6.82   20-Feb-26
    Conventions:  CF-1.0


In [6]:
# Look at just the RAINFALL variable in detail
print("Shape  :", ds['RAINFALL'].shape)
print("Dims   :", ds['RAINFALL'].dims)
print("Dtype  :", ds['RAINFALL'].dtype)
print("Min    :", float(ds['RAINFALL'].min()))
print("Max    :", float(ds['RAINFALL'].max()))
print("NaN count:", int(ds['RAINFALL'].isnull().sum()))

Shape  : (365, 129, 135)
Dims   : ('TIME', 'LATITUDE', 'LONGITUDE')
Dtype  : float32
Min    : 0.0
Max    : 747.7235717773438
NaN count: 4544615


In [7]:
# Goa bounding box
GOA_LAT_MIN, GOA_LAT_MAX = 14.9, 15.7
GOA_LON_MIN, GOA_LON_MAX = 73.7, 74.3

# Slice the dataset to Goa region only
goa = ds['RAINFALL'].sel(
    LATITUDE  = slice(GOA_LAT_MIN, GOA_LAT_MAX),
    LONGITUDE = slice(GOA_LON_MIN, GOA_LON_MAX)
)

print("Goa slice shape:", goa.shape)
print("Latitude points :", goa.LATITUDE.values)
print("Longitude points:", goa.LONGITUDE.values)
print("\nFirst 10 days of spatially averaged Goa rainfall (mm/day):")

# Average across all Goa grid points (ignoring NaN = ocean pixels)
goa_daily = goa.mean(dim=['LATITUDE','LONGITUDE'], skipna=True)
print(goa_daily.values[:10])

Goa slice shape: (365, 3, 3)
Latitude points : [15.   15.25 15.5 ]
Longitude points: [73.75 74.   74.25]

First 10 days of spatially averaged Goa rainfall (mm/day):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [8]:
# Check July 1991 — should show heavy rainfall
july_mask = ds['TIME'].dt.month == 7
goa_july = goa_daily.sel(TIME=july_mask)

print("July 1991 daily rainfall (mm/day):")
for date, val in zip(goa_july.TIME.values, goa_july.values):
    print(f"  {str(date)[:10]}  :  {val:.1f} mm")

print(f"\nJuly 1991 total : {float(goa_july.sum()):.1f} mm")
print(f"July 1991 max   : {float(goa_july.max()):.1f} mm")

July 1991 daily rainfall (mm/day):
  1991-07-01  :  16.7 mm
  1991-07-02  :  24.1 mm
  1991-07-03  :  23.4 mm
  1991-07-04  :  27.5 mm
  1991-07-05  :  45.5 mm
  1991-07-06  :  30.7 mm
  1991-07-07  :  63.6 mm
  1991-07-08  :  81.5 mm
  1991-07-09  :  75.6 mm
  1991-07-10  :  39.5 mm
  1991-07-11  :  35.2 mm
  1991-07-12  :  47.7 mm
  1991-07-13  :  79.5 mm
  1991-07-14  :  54.1 mm
  1991-07-15  :  90.7 mm
  1991-07-16  :  118.9 mm
  1991-07-17  :  151.2 mm
  1991-07-18  :  96.8 mm
  1991-07-19  :  52.4 mm
  1991-07-20  :  50.8 mm
  1991-07-21  :  54.3 mm
  1991-07-22  :  61.4 mm
  1991-07-23  :  77.1 mm
  1991-07-24  :  50.4 mm
  1991-07-25  :  55.2 mm
  1991-07-26  :  97.3 mm
  1991-07-27  :  101.9 mm
  1991-07-28  :  82.1 mm
  1991-07-29  :  67.7 mm
  1991-07-30  :  63.0 mm
  1991-07-31  :  55.8 mm

July 1991 total : 1971.6 mm
July 1991 max   : 151.2 mm
